# Electivo de Bioinformática — Clase 5

## Mejoramiento de archivos BAM, scripts y workflows

**Programa:** Doctorado — 2º año
**Duración:** 3 horas
**Fecha:** 22 de septiembre

### Objetivos de la clase

Al finalizar esta clase, serán capaces de:

1. Explicar qué significa "mejorar" (refinar) un BAM y en qué punto del análisis se ubica este paso: entre el alineamiento y el llamado de variantes.
2. Instalar **Picard** y **GATK4** (con conda y de forma manual) y verificar que funcionan.
3. Usar **Picard `FixMateInformation`** para sincronizar la información de las parejas de lecturas (*mates*) y agregar la etiqueta `MC`, comprobando el antes y el después con `ValidateSamFile`.
4. Marcar lecturas duplicadas con **Picard `MarkDuplicates`**, interpretar sus métricas y saber cuándo **no** hacerlo.
5. Explicar y ejecutar la **recalibración de calidades base (BQSR)** con GATK4 (`BaseRecalibrator` + `ApplyBQSR`) e interpretar la tabla de recalibración.
6. Definir qué es un **workflow** (flujo de trabajo) y distinguir un script de un gestor de workflows (Snakemake, Nextflow, WDL).
7. Aplicar **buenas prácticas** para escribir un script reproducible que alinee y mejore un BAM, y ejecutarlo sobre una muestra nueva.

---
## 1. Repaso y panorama: ¿qué es "mejorar" un BAM? (10 min)

En la clase 4 dejamos un BAM **ordenado e indexado**. Ese BAM es correcto, pero todavía es un BAM "crudo": el alineador tomó decisiones lectura por lectura y no revisó que el conjunto sea coherente. Los llamadores de variantes, en cambio, **asumen** que el BAM es coherente. El paso de *mejoramiento* (en la literatura, *pre-processing* o preparación de un *analysis-ready BAM*) cierra esa brecha:

```
FASTQ ─► QC ─► bwa mem ─► sort ─► FixMateInformation ─► MarkDuplicates ─► BQSR ─► BAM listo ─► variantes
        (clase 4)  (clase 4)         (hoy)              (hoy)             (hoy)              (próx. clase)
```

| Paso | Problema que resuelve | Herramienta |
|---|---|---|
| **FixMateInformation** | Que cada lectura y su pareja "se pongan de acuerdo" sobre dónde está cada una (posición, hebra, tamaño de inserto, CIGAR) | Picard |
| **MarkDuplicates** | Lecturas duplicadas (PCR/óptico) que contarían la misma molécula varias veces | Picard |
| **BQSR** | Las calidades base que reporta el secuenciador tienen sesgos sistemáticos | GATK |

Hoy practicamos los tres pasos en negrita, en el orden en que los recomienda GATK (`MarkDuplicates` va **entre** `FixMateInformation` y BQSR). Después de practicar cada paso "a mano" aprenderemos a **automatizarlos** correctamente: eso es un *workflow*.

> **Nota sobre mtDNA.** Seguimos usando el genoma mitocondrial humano (rCRS) por rapidez y continuidad con la clase 4. Ojo con una diferencia importante: BQSR fue diseñado para datos nucleares humanos, donde existen catálogos de variantes conocidas (dbSNP y otros). La receta publicada de GATK para llamar variantes en mtDNA (Mutect2 en modo mitocondrial) no incluye BQSR. Aquí lo usamos porque es una excelente forma de **entender cómo funciona**, no porque sea la práctica estándar para mtDNA.

---
## 2. Instalación de Picard y GATK4 (20 min)

### 2.1 Requisitos previos

Picard y GATK4 están escritos en **Java**. GATK 4.4 en adelante requiere **Java 17** (versiones más nuevas suelen funcionar, pero la oficialmente soportada es la 17). Con conda no hay que instalar Java a mano: se instala automáticamente como dependencia dentro del ambiente.

### 2.2 Método recomendado: conda (bioconda)

Siguiendo lo que hicimos en la clase 3, creamos un **ambiente nuevo** para este flujo. ¿Por qué no simplemente agregar Picard y GATK al ambiente `alineamiento`? Porque GATK4 trae sus propias dependencias (Java 17, Python, muchas librerías) y mezclarlas con otras herramientas aumenta las probabilidades de conflictos que el solucionador de conda no pueda resolver. **Un ambiente por flujo de trabajo** también facilita la reproducibilidad: el ambiente `mejoramiento` contendrá exactamente lo necesario para el script de hoy.

Instalamos: `bwa` y `samtools` (para alinear), `picard`, `gatk4`, y `dwgsim` (solo por si hay que regenerar los datos de la clase 4).

> **Tip:** ejecuten esta celda **ahora** y sigan leyendo mientras corre. El paquete `picard` de bioconda incluye R (se usa solo para algunos gráficos de métricas), así que la primera instalación puede tardar varios minutos.

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"

echo "--- Creando el ambiente 'mejoramiento' (bwa, samtools, picard, gatk4, dwgsim) ---"
conda create -y -n mejoramiento -c conda-forge -c bioconda bwa samtools picard gatk4 dwgsim

echo
echo "--- Ambientes disponibles ---"
conda env list

Verificamos la instalación. Cada celda `%%bash` es una shell nueva, así que en **cada** celda hay que repetir `source .../conda.sh` y `conda activate mejoramiento` (lo mismo que en la clase 4).

Dos detalles que conviene notar en la salida:

- `picard` es un *wrapper* (un script pequeño que ejecuta `java -jar picard.jar ...` por nosotros).
- **GATK4 trae Picard incorporado.** Fíjense en la lista de herramientas: aparecen `FixMateInformation (Picard)`, `MarkDuplicates (Picard)` y `ValidateSamFile (Picard)` *dentro de* `gatk`. Por eso en tutoriales verán a veces `gatk FixMateInformation ...` y otras `picard FixMateInformation ...`: es la misma herramienta. Hoy usaremos `picard` para los pasos de Picard y `gatk` para BQSR, para dejar claro de dónde viene cada uno.

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

echo "--- Java ---"
java -version 2>&1 | head -1

echo
echo "--- Picard ---"
picard FixMateInformation --version 2>&1 | tail -1

echo
echo "--- GATK ---"
gatk --version 2>&1 | grep -E "Genome Analysis|HTSJDK|Picard"

echo
echo "--- Herramientas de hoy dentro de GATK4 ---"
gatk --list 2>&1 | sed 's/\x1b\[[0-9;]*m//g' | grep -E "^ +(FixMateInformation|MarkDuplicates|ValidateSamFile|BaseRecalibrator|ApplyBQSR) "

### 2.3 Método alternativo: instalación manual (sin conda)

Sirve cuando **no pueden usar conda** (por ejemplo, un clúster con políticas restrictivas) o cuando necesitan una versión específica. Solo requiere Java 17 o superior (`java -version`). **No ejecutaremos estas celdas en clase** (GATK pesa cerca de 700 MB), pero es importante que sepan hacerlo.

**Picard** es un único archivo `.jar`:

```bash
PICARD_V=3.4.0        # ajusten a la version mas reciente: github.com/broadinstitute/picard/releases
mkdir -p ~/software/picard && cd ~/software/picard
wget https://github.com/broadinstitute/picard/releases/download/${PICARD_V}/picard.jar

# Se ejecuta con java -jar (el nombre de la herramienta va justo despues del .jar):
java -Xmx4g -jar ~/software/picard/picard.jar FixMateInformation --help
```

**GATK4** viene en un `.zip` que incluye un script `gatk` (el *wrapper* que llama a Java por nosotros):

```bash
GATK_V=4.6.2.0        # ajusten a la version mas reciente: github.com/broadinstitute/gatk/releases
mkdir -p ~/software && cd ~/software
wget https://github.com/broadinstitute/gatk/releases/download/${GATK_V}/gatk-${GATK_V}.zip
unzip gatk-${GATK_V}.zip

# Para poder escribir simplemente "gatk" desde cualquier carpeta, lo agregamos al PATH
# (recuerdan clase 2: variable PATH y ~/.bashrc):
echo 'export PATH="$HOME/software/gatk-'${GATK_V}':$PATH"' >> ~/.bashrc
source ~/.bashrc
gatk --version
```

Para que `picard` también se pueda llamar con una sola palabra, pueden crear un alias: `alias picard='java -Xmx4g -jar ~/software/picard/picard.jar'` (también en `~/.bashrc`).

Notas: (a) el zip de GATK incluye un ambiente conda opcional (`gatkcondaenv.yml`) que solo es necesario para unas pocas herramientas basadas en Python, que no usaremos; (b) también existe una **imagen Docker** oficial (`broadinstitute/gatk`), muy usada en servidores y en la nube porque garantiza que todos ejecutan exactamente la misma versión.

### 2.4 Cómo se invoca cada herramienta

| | Picard | GATK4 |
|---|---|---|
| Sintaxis | `picard Herramienta -I in.bam -O out.bam` | `gatk Herramienta -I in.bam -O out.bam` |
| Con el .jar manual | `java -jar picard.jar Herramienta ...` | `./gatk Herramienta ...` |
| Memoria de Java | `picard -Xmx4g Herramienta ...` | `gatk --java-options "-Xmx4g" Herramienta ...` |
| Argumentos | `--NOMBRE_EN_MAYUSCULA valor` (con atajos: `-I`, `-O`, `-R`). Las versiones antiguas usaban `I=in.bam` | `--nombre-en-minuscula valor` (con atajos: `-I`, `-O`, `-R`) |
| Ayuda | `picard Herramienta --help` | `gatk Herramienta --help` |

Los dos usan la convención de que **los argumentos largos llevan doble guion**. Si un tutorial antiguo usa `I=archivo.bam`, es la sintaxis vieja de Picard: en versiones recientes conviene usar la nueva.

---
## 3. Preparación de los datos de partida (10 min)

Vamos a trabajar en una carpeta nueva, `~/bioinfo/clase5/`. Como punto de partida usamos **los mismos datos de la clase 4** (genoma mitocondrial rCRS y las 2000 parejas de lecturas simuladas con `dwgsim`). Si por algún motivo ya no tienen esos archivos, la celda los regenera con los mismos parámetros y la misma semilla (`-z 7`), así que el resultado es idéntico.

In [ ]:
%%bash
mkdir -p ~/bioinfo/clase5/{referencia,fastq,alineamiento,scripts}
cd ~/bioinfo/clase5

C4=~/bioinfo/clase4
if [ -s $C4/referencia/chrM.fasta ] && [ -s $C4/fastq/mtDNA_sample_R1.fastq.gz ] \
   && [ -s $C4/fastq/mtDNA_sample_R2.fastq.gz ] && [ -s $C4/fastq/mtDNA_sample.mutations.vcf ]; then
    echo "--- Usando los archivos que dejamos listos en la clase 4 ---"
    cp $C4/referencia/chrM.fasta referencia/
    cp $C4/fastq/mtDNA_sample_R1.fastq.gz $C4/fastq/mtDNA_sample_R2.fastq.gz fastq/
    cp $C4/fastq/mtDNA_sample.mutations.vcf fastq/
else
    echo "--- No encontre los archivos de la clase 4: los regenero ---"
    source "$HOME/miniforge3/etc/profile.d/conda.sh"
    conda activate mejoramiento

    URL="https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id=NC_012920.1&rettype=fasta&retmode=text"
    wget -q -O referencia/chrM.fasta "$URL" && grep -q "^>" referencia/chrM.fasta || { echo "ERROR: no se pudo descargar la referencia"; exit 1; }

    cd fastq
    dwgsim -N 2000 -1 100 -2 100 -e 0.001 -E 0.001 -r 0.0006 -R 0 -X 0 -y 0 -z 7 -c 0 \
        ../referencia/chrM.fasta mtDNA_sample 2> dwgsim.log
    mv mtDNA_sample.bwa.read1.fastq.gz mtDNA_sample_R1.fastq.gz
    mv mtDNA_sample.bwa.read2.fastq.gz mtDNA_sample_R2.fastq.gz
    rm -f mtDNA_sample.bfast.fastq.gz
fi

cd ~/bioinfo/clase5
echo
ls -la referencia fastq

### 3.1 Índices de la referencia: ahora necesitamos uno más

En la clase 4 creamos dos índices: el de `bwa` (para alinear) y el `.fai` de `samtools` (para acceso rápido). **GATK y Picard exigen un tercero: el *diccionario de secuencias* (`.dict`)**, un pequeño archivo con el nombre, el largo y el checksum (MD5) de cada contig de la referencia. Se usa para comprobar que el BAM, la referencia y los VCF hablan *exactamente* del mismo genoma. Si falta, GATK se detiene con un error que menciona "sequence dictionary".

| Archivo | Lo crea | Lo usa |
|---|---|---|
| `chrM.fasta.{amb,ann,bwt,pac,sa}` | `bwa index` | `bwa mem` |
| `chrM.fasta.fai` | `samtools faidx` | `samtools`, GATK |
| `chrM.dict` | `picard CreateSequenceDictionary` (o `gatk CreateSequenceDictionary`) | Picard, GATK |

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase5/referencia

echo "--- Indice .fai (samtools) ---"
samtools faidx chrM.fasta
cat chrM.fasta.fai

echo
echo "--- Indice de bwa ---"
bwa index chrM.fasta 2>&1 | tail -2

echo
echo "--- Diccionario de secuencias (Picard) ---"
picard CreateSequenceDictionary -R chrM.fasta -O chrM.dict 2> picard_dict.log || cat picard_dict.log
cat chrM.dict

echo
ls -1

### 3.2 Alineamiento de partida

Repetimos el alineamiento de la clase 4, con una diferencia práctica: **encadenamos** `bwa mem` y `samtools sort` con un *pipe* (`|`). Así no se escribe en disco el SAM intermedio (que es grande y solo estorba). También agregamos `LB` (librería) al read group, que Picard usa más adelante para marcar duplicados por librería.

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase5

bwa mem -t 2 -R '@RG\tID:mtDNA_sample\tSM:mtDNA_sample\tLB:lib1\tPL:ILLUMINA' \
    referencia/chrM.fasta \
    fastq/mtDNA_sample_R1.fastq.gz fastq/mtDNA_sample_R2.fastq.gz 2> alineamiento/bwa.log \
  | samtools sort -O bam -o alineamiento/mtDNA_sample.sorted.bam -

samtools index alineamiento/mtDNA_sample.sorted.bam

echo "--- flagstat ---"
samtools flagstat alineamiento/mtDNA_sample.sorted.bam | head -5
ls -la alineamiento

---
## 4. Picard `FixMateInformation` (25 min)

### 4.1 ¿Qué es la "información de la pareja" (*mate information*)?

En una secuenciación pareada, cada fragmento produce dos lecturas (R1 y R2). En el BAM son **dos registros**, y **cada uno guarda datos sobre el otro** en sus campos:

| Campo / etiqueta | Qué guarda sobre la pareja |
|---|---|
| `RNEXT` (col. 7) | Contig donde alineó la pareja (`=` si es el mismo) |
| `PNEXT` (col. 8) | Posición donde alineó la pareja |
| `TLEN` (col. 9) | Tamaño del inserto (positivo en una lectura, negativo en la otra) |
| `FLAG` (bits `0x8` y `0x20`) | ¿La pareja está sin alinear? ¿Alineó en la hebra reversa? |
| `MC` (etiqueta) | **CIGAR** de la pareja |
| `MQ` (etiqueta) | Calidad de mapeo (MAPQ) de la pareja |

Miremos una pareja real de nuestro BAM. Los dos registros deben ser **espejos**: la `POS` de uno es el `PNEXT` del otro, y el `TLEN` cambia de signo.

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase5/alineamiento

# Tomamos el nombre de la primera lectura y buscamos a sus dos registros (R1 y R2)
NOMBRE=$(samtools view mtDNA_sample.sorted.bam | head -1 | cut -f1)
echo "Pareja: $NOMBRE"
echo
echo -e "FLAG\tRNAME\tPOS\tMAPQ\tCIGAR\tRNEXT\tPNEXT\tTLEN\t(etiquetas...)"
samtools view mtDNA_sample.sorted.bam | awk -v n="$NOMBRE" '$1==n' | cut -f2-9,12-

### 4.2 ¿Cuándo se desincroniza esta información?

Un alineador como `bwa mem` **escribe la información de las parejas bien**, porque ve ambas lecturas a la vez. El problema aparece cuando **otra herramienta modifica un registro sin actualizar el de su pareja**. Casos típicos:

- Herramientas que **realinean** o mueven lecturas de forma independiente (el clásico: `IndelRealigner` de GATK 3, que dejaba a las parejas desincronizadas y hacía necesario correr `FixMateInformation` después).
- Cuando R1 y R2 se **alinean por separado** (como si fueran lecturas sin pareja) y luego se "pegan" con un script propio, o cuando se **unen/dividen BAMs**.
- **Filtros** que eliminan solo una de las dos lecturas, dejando a la otra apuntando a una pareja que ya no existe.
- Ediciones manuales o *scripts* propios sobre el BAM.

Además, `FixMateInformation` agrega la etiqueta **`MC`** (CIGAR de la pareja). Con `MC`, una herramienta sabe dónde termina la pareja *sin tener que ir a buscar su registro*, y algunas la **exigen** (por ejemplo, `MarkDuplicatesWithMateCigar` de Picard; el `MarkDuplicates` estándar, en cambio, encuentra a las parejas por nombre y no la necesita). Como regla práctica: se corre `FixMateInformation` **justo después de cualquier paso que modifique lecturas**, para que todo lo que venga después reciba un BAM coherente.

### 4.3 Recreemos el problema (a propósito)

Como nuestro BAM viene "sano" de `bwa`, vamos a **dañarlo artificialmente** para imitar lo que haría una de esas herramientas. Con `awk` modificamos dos cosas: en 1 de cada 10 lecturas alteramos `PNEXT` (posición de la pareja), y en 1 de cada 7 borramos la etiqueta `MC`. Lo importante no es el `awk`, sino comprobar que **`ValidateSamFile`** de Picard detecta el daño.

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase5/alineamiento

# BAM -> texto SAM -> danar -> BAM
samtools view -h mtDNA_sample.sorted.bam \
  | awk 'BEGIN{FS=OFS="\t"}
         /^@/ {print; next}
         { n++
           if (n % 10 == 0) $8 = $8 + 15          # PNEXT incorrecto
           if (n % 7 == 0) {                      # sin etiqueta MC
               linea = $1
               for (i = 2; i <= NF; i++) if ($i !~ /^MC:Z:/) linea = linea OFS $i
               print linea
           } else print }' \
  | samtools view -b -o mtDNA_sample.danado.bam -
samtools index mtDNA_sample.danado.bam

echo "--- Lecturas totales: $(samtools view -c mtDNA_sample.danado.bam) ---"
echo "--- Lecturas con etiqueta MC: bueno=$(samtools view mtDNA_sample.sorted.bam | grep -c 'MC:Z:')  danado=$(samtools view mtDNA_sample.danado.bam | grep -c 'MC:Z:') ---"

echo
echo "--- ValidateSamFile sobre el BAM danado ---"
picard -Xmx2g ValidateSamFile -I mtDNA_sample.danado.bam -R ../referencia/chrM.fasta --MODE SUMMARY 2>&1 \
  | grep -E "ERROR|WARNING:|No errors"

`ValidateSamFile --MODE SUMMARY` cuenta los errores por tipo. Deberían ver ≈400 errores `MISMATCH_MATE_ALIGNMENT_START`: "la posición de la pareja que dice este registro no coincide con la posición real de la pareja". (Las ≈570 lecturas sin `MC` no cuentan como *error*, pero sí nos faltan datos que otras herramientas quieren usar.)

### 4.4 Ejecutar `FixMateInformation`

```
picard FixMateInformation -I entrada.bam -O salida.bam --ADD_MATE_CIGAR true
```

| Argumento | Para qué sirve | Valor por defecto |
|---|---|---|
| `-I` / `--INPUT` | BAM de entrada (puede ser más de uno: los une y ordena) | obligatorio |
| `-O` / `--OUTPUT` | BAM de salida. **Si se omite, sobrescribe la entrada** (dejando una copia `.old`), así que en general conviene siempre indicarlo | (sobrescribe) |
| `--ADD_MATE_CIGAR` (`-MC`) | Agrega/actualiza la etiqueta `MC` | `true` |
| `--IGNORE_MISSING_MATES` | Si una pareja falta en el archivo (por ejemplo por un filtro previo), se ignora en vez de terminar con error | `true` |
| `--SORT_ORDER` (`-SO`) | Ordenar la salida de otra forma que la entrada (`coordinate`, `queryname`...) | conserva el orden de la entrada |
| `--ASSUME_SORTED` (`-AS`) | Tratar la entrada como ordenada por nombre aunque el encabezado diga otra cosa | `false` |

**¿Cómo lo hace?** Para comparar cada lectura con su pareja, Picard necesita tenerlas *juntas*, por eso internamente **ordena por nombre de lectura** (*queryname*), recorre las parejas corrigiendo los campos, y al final **vuelve a ordenar por coordenada** para entregarnos el BAM en el mismo orden en que lo recibió. Esto es lo que significan los mensajes de su log ("Sorting by queryname", "re-sorting output file"). Es rápido en nuestros datos, pero en un genoma completo es un paso costoso en tiempo y disco temporal.

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase5/alineamiento

picard -Xmx2g FixMateInformation \
    -I mtDNA_sample.danado.bam \
    -O mtDNA_sample.fixmate.bam \
    --ADD_MATE_CIGAR true 2>&1 | grep -E "FixMateInformation|Sorting|sorted|re-sorting" | cut -c1-140

samtools index mtDNA_sample.fixmate.bam
echo
ls -la mtDNA_sample.fixmate.bam*

### 4.5 Comprobar que quedó arreglado

Tres verificaciones: (1) `ValidateSamFile` debe decir `No errors found`; (2) todas las lecturas deben tener de nuevo `MC`; (3) como sabemos cuál era el BAM original sano, podemos comprobar que los campos de las parejas quedaron **idénticos** a los originales (comparando un *checksum* de las columnas 1 a 9 de todas las lecturas).

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase5/alineamiento

echo "--- 1) ValidateSamFile sobre el BAM arreglado ---"
picard -Xmx2g ValidateSamFile -I mtDNA_sample.fixmate.bam -R ../referencia/chrM.fasta --MODE SUMMARY 2>&1 \
  | grep -E "ERROR|WARNING:|No errors"

echo
echo "--- 2) Lecturas con etiqueta MC (de $(samtools view -c mtDNA_sample.fixmate.bam)) ---"
samtools view mtDNA_sample.fixmate.bam | grep -c 'MC:Z:'

echo
echo "--- 3) Checksum de las columnas 1-9 (nombre, flag, posicion, ..., PNEXT, TLEN) ---"
for b in sorted danado fixmate; do
    printf "%-8s " $b
    samtools view mtDNA_sample.$b.bam | cut -f1-9 | sort | md5sum | cut -c1-12
done

`sorted` (el original sano) y `fixmate` deben tener **el mismo checksum**, y `danado` uno distinto: `FixMateInformation` recuperó exactamente la información que nos habíamos "roto". Además agregó la etiqueta `MQ` (calidad de mapeo de la pareja) y devolvió `MC` a todas las lecturas.

> **Para discutir (2 min, en parejas):** (a) Si un filtro dejó lecturas *huérfanas* (sin su pareja en el archivo), ¿qué hace `FixMateInformation` con ellas y qué parámetro lo controla? (b) ¿Por qué conviene ejecutar `FixMateInformation` justo después del paso que modificó las lecturas, y no al final de todo el flujo?

> **Alternativa en `samtools`:** existe `samtools fixmate` (con `-m` agrega la información que necesita `samtools markdup`), pero exige que el BAM esté ordenado **por nombre**. El flujo completo en `samtools` es `sort -n` → `fixmate -m` → `sort` → `markdup`. Nosotros usamos la versión de Picard porque es la del flujo recomendado por GATK y no requiere que ordenemos nosotros por nombre.

---
## 5. Picard `MarkDuplicates` (20 min)

### 5.1 ¿Qué son los duplicados y por qué molestan?

Al preparar una librería, el ADN se **amplifica por PCR**. Si una misma molécula original se amplifica y se secuencia varias veces, obtenemos **varias parejas de lecturas casi idénticas que en realidad son una sola observación**. Hay dos orígenes:

- **Duplicados de PCR** (de la preparación de la librería): copias de la misma molécula.
- **Duplicados ópticos** (de la secuenciación): un mismo *cluster* de la celda de flujo detectado como dos clusters vecinos.

¿Por qué importan para las variantes?

1. **Inflan la cobertura**: 30 lecturas que en realidad son 10 moléculas independientes dan una falsa sensación de respaldo.
2. **Multiplican los errores de PCR**: un error introducido en los primeros ciclos de amplificación se copia a todos sus descendientes y aparece en muchas lecturas, *parece* una variante bien respaldada (falso positivo).
3. **Distorsionan la fracción alélica**: y con ella la distinción entre homocigoto/heterocigoto o, en mtDNA, el porcentaje de heteroplasmia.

### 5.2 ¿Cómo los detecta Picard?

`MarkDuplicates` considera duplicadas a las **parejas cuyos extremos 5' (sin contar *soft-clips*) y orientación coinciden en las dos lecturas**, dentro de la **misma librería** (etiqueta `LB` del *read group*: por eso la agregamos al alinear). De cada grupo de duplicados conserva como "original" a la pareja con mayor suma de calidades base, y a las demás les **marca el flag `0x400` (1024)**. Por defecto **no las elimina**: quedan en el BAM, marcadas, y las herramientas posteriores (por ejemplo `BaseRecalibrator`, como veremos) las saltan. La entrada debe estar **ordenada por coordenada**.

| Argumento | Para qué sirve | Por defecto |
|---|---|---|
| `-I` / `--INPUT` | BAM ordenado por coordenada | obligatorio |
| `-O` / `--OUTPUT` | BAM de salida con los duplicados marcados | obligatorio |
| `-M` / `--METRICS_FILE` | Archivo de **métricas** de duplicación (¡es obligatorio!) | obligatorio |
| `--REMOVE_DUPLICATES` | Si es `true`, **elimina** los duplicados en vez de solo marcarlos | `false` |
| `--DUPLICATE_SCORING_STRATEGY` (`-DS`) | Cómo elegir a la pareja "original" del grupo | `SUM_OF_BASE_QUALITIES` |
| `--OPTICAL_DUPLICATE_PIXEL_DISTANCE` | Distancia máxima entre clusters para considerarlos duplicados ópticos. El valor por defecto sirve para celdas de flujo *no patrón*; para modelos con celda *patterned* (NovaSeq, NextSeq...) la ayuda de Picard recomienda 2500 | 100 |
| `--READ_NAME_REGEX` | Cómo leer *tile*, *x* e *y* desde el nombre de la lectura para detectar duplicados ópticos. Con `null` se desactiva esa detección | (formato Illumina) |

> **¿Cuándo NO marcar duplicados?** En **amplicones o PCR de largo alcance** (por ejemplo, mtDNA amplificado en dos fragmentos largos, o paneles de amplicones): por diseño casi todas las lecturas comienzan y terminan en las mismas posiciones, y `MarkDuplicates` marcaría como "duplicada" cobertura perfectamente legítima. En captura y genoma completo sí conviene.

### 5.3 Ejecutar `MarkDuplicates` sobre nuestro BAM

Partimos del BAM que dejó `FixMateInformation` en la sección 4 (`mtDNA_sample.fixmate.bam`, que ya sabemos que está sano y ordenado por coordenada). `MarkDuplicates` **no crea el índice** del BAM de salida, así que lo indexamos con `samtools`.

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase5/alineamiento

picard -Xmx2g MarkDuplicates \
    -I mtDNA_sample.fixmate.bam \
    -O mtDNA_sample.markdup.bam \
    -M mtDNA_sample.dup_metrics.txt 2> markdup.log || cat markdup.log

samtools index mtDNA_sample.markdup.bam

echo "--- Resumen del log ---"
grep -E "Marking|duplicate clusters" markdup.log | cut -f4- | cut -c1-120

echo
echo "--- Metricas (columnas principales) ---"
grep -A1 '^LIBRARY' mtDNA_sample.dup_metrics.txt | cut -f1,3,7-10

echo
echo "--- Lecturas marcadas como duplicado (samtools flagstat) ---"
echo "antes:   $(samtools flagstat mtDNA_sample.fixmate.bam | grep -m1 duplicates)"
echo "despues: $(samtools flagstat mtDNA_sample.markdup.bam | grep -m1 duplicates)"

### 5.4 Cómo leer las métricas y ver qué se marcó

Las columnas principales del archivo de métricas son:

| Métrica | Significado |
|---|---|
| `READ_PAIRS_EXAMINED` | Parejas de lecturas analizadas (aquí 2000) |
| `READ_PAIR_DUPLICATES` | Parejas marcadas como duplicado (¡no lecturas!: cada pareja son 2 lecturas) |
| `READ_PAIR_OPTICAL_DUPLICATES` | De ellas, las que Picard atribuye a duplicación óptica |
| `PERCENT_DUPLICATION` | Fracción de lecturas duplicadas (0,001 = 0,1 %) |
| `ESTIMATED_LIBRARY_SIZE` | Estimación del número de moléculas distintas de la librería (con muy pocos duplicados, como en nuestra primera muestra, es poco confiable) |

Con nuestros datos simulados **no hubo PCR**, y aun así aparecen unos pocos duplicados: con 2.000 parejas cayendo al azar sobre solo 16,5 kb del genoma, algunas coinciden en las mismas posiciones **por casualidad**. (El pequeño número de "ópticos" es un artefacto: los nombres de lectura de `dwgsim` no tienen formato Illumina y Picard interpreta partes del nombre como coordenadas.)

Miremos las lecturas marcadas. Un duplicado tiene el bit `0x400` (1024) encendido en el FLAG; por ejemplo un `99` (pareja, propia, primera de la pareja) pasa a `99 + 1024 = 1123`:

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase5/alineamiento

echo "--- Lecturas con el flag 1024 (samtools view -f 1024) ---"
echo -e "NOMBRE\tFLAG\tRNAME\tPOS\tMAPQ\tCIGAR\tRNEXT\tPNEXT\tTLEN"
samtools view -f 1024 mtDNA_sample.markdup.bam | cut -f1-9

echo
echo "--- Que significa un flag 1123 ---"
samtools flags 1123

echo
echo "--- Conteo: total / duplicadas / no duplicadas ---"
echo "total:        $(samtools view -c mtDNA_sample.markdup.bam)"
echo "duplicadas:   $(samtools view -c -f 1024 mtDNA_sample.markdup.bam)"
echo "no duplicadas: $(samtools view -c -F 1024 mtDNA_sample.markdup.bam)"

### 5.5 Un caso con "PCR": duplicados de verdad

Para ver a `MarkDuplicates` trabajando de verdad, vamos a **fabricar una librería con PCR excesiva**: tomamos nuestro FASTQ y le agregamos una **copia de las primeras 300 parejas** (con otros nombres, para que sean lecturas "distintas" que salen de la misma molécula). Debería detectar aproximadamente esas 300 parejas, más las 2 de casualidad de antes. Como esta vez sí son duplicados masivos, desactivamos la detección óptica (`--READ_NAME_REGEX null`) porque los nombres simulados no tienen coordenadas de *tile*.

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase5
mkdir -p duplicados_demo && cd duplicados_demo

# FASTQ = original + copia de las primeras 300 parejas (1200 lineas) con nombre modificado (_pcr)
for R in 1 2; do
    ( zcat ../fastq/mtDNA_sample_R$R.fastq.gz
      zcat ../fastq/mtDNA_sample_R$R.fastq.gz | head -n 1200 | sed -E '1~4 s#^(@[^ ]+)/([12])$#\1_pcr/\2#'
    ) | gzip > mtDNA_pcr_R$R.fastq.gz
done
echo "Parejas en el FASTQ con PCR: $(( $(zcat mtDNA_pcr_R1.fastq.gz | wc -l) / 4 ))"

bwa mem -t 2 -R '@RG\tID:mtDNA_pcr\tSM:mtDNA_pcr\tLB:lib1\tPL:ILLUMINA' \
    ../referencia/chrM.fasta mtDNA_pcr_R1.fastq.gz mtDNA_pcr_R2.fastq.gz 2> bwa.log \
  | samtools sort -O bam -o mtDNA_pcr.sorted.bam -

picard -Xmx2g MarkDuplicates \
    -I mtDNA_pcr.sorted.bam -O mtDNA_pcr.markdup.bam \
    -M mtDNA_pcr.dup_metrics.txt \
    --READ_NAME_REGEX null 2> markdup.log || cat markdup.log

echo
echo "--- Metricas ---"
grep -A1 '^LIBRARY' mtDNA_pcr.dup_metrics.txt | cut -f1,3,7-10

echo
echo "--- Lecturas: total / duplicadas ---"
echo "total:      $(samtools view -c mtDNA_pcr.markdup.bam)"
echo "duplicadas: $(samtools view -c -f 1024 mtDNA_pcr.markdup.bam)"

Deberían ver ≈302 parejas duplicadas (300 que "fabricamos" + 2 de casualidad) de 2.300 examinadas: `PERCENT_DUPLICATION` ≈ 0,13. Fíjense en un detalle importante: los nombres de las copias son **distintos** a los originales, y aun así Picard las detectó. **No mira los nombres: mira dónde comienzan y terminan las dos lecturas de cada pareja.**

> **Para discutir (2 min, en parejas):** (a) Si se hubiera usado `--REMOVE_DUPLICATES true`, ¿cuántas lecturas tendría el BAM? ¿Qué ventaja tiene *marcar* en vez de *eliminar*? (b) Si esta librería fuera un amplicón de mtDNA de 3 kb en vez de un genoma completo, ¿qué pasaría con `PERCENT_DUPLICATION`, y qué decisión tomarían?

**Desde ahora, el BAM de trabajo es `mtDNA_sample.markdup.bam`** (el de la sección 5.3, no el de la demostración). Es el que usaremos para BQSR.

---
## 6. Recalibración de calidades base: BQSR con GATK4 (35 min)

### 6.1 Repaso: la calidad de una base (Phred)

Cada base de una lectura viene con una calidad **Q** (columna 11 del SAM / cuarta línea del FASTQ), que expresa la probabilidad de que esa base esté mal llamada:

$$Q = -10 \cdot \log_{10}(P_{error})$$

| Q | Probabilidad de error | Exactitud |
|---|---|---|
| 10 | 1 en 10 | 90 % |
| 20 | 1 en 100 | 99 % |
| 30 | 1 en 1.000 | 99,9 % |
| 40 | 1 en 10.000 | 99,99 % |

Esa Q la **estima el secuenciador**, y es una estimación con sesgos: puede sobre- o subestimar el error real según el ciclo de secuenciación, el contexto de nucleótidos, la lane, etc. Los llamadores de variantes usan esas calidades para ponderar la evidencia (una base Q40 "vota" mucho más que una Q10), así que si están mal calibradas, **las variantes se llaman peor**.

### 6.2 La idea de BQSR

*Base Quality Score Recalibration* corrige esos sesgos en dos pasos:

1. **`BaseRecalibrator`** construye un modelo de error. Recorre todas las bases del BAM y, en cada una, compara con la referencia. Agrupa las bases por **covariables** y cuenta, para cada grupo, cuántas veces no coincidieron con la referencia. Las covariables son: **grupo de lectura** (*read group*), **calidad reportada**, **ciclo** (posición en la lectura) y **contexto** (la base actual y la anterior).
2. **`ApplyBQSR`** usa esa tabla para **reescribir** la calidad de cada base según el error que *realmente* se observó en su grupo.

Hay un supuesto clave: *todo desacuerdo con la referencia es un error de secuenciación*. Como eso no es cierto en los sitios donde la muestra tiene una variante real, hay que **enmascarar esos sitios** entregando una lista de variantes conocidas (`--known-sites`). En humano se usan catálogos como dbSNP y los conjuntos de indels de Mills/1000 Genomas (parte del *resource bundle* de GATK). Si su organismo no tiene esos catálogos, la estrategia sugerida por GATK es *bootstrapping*: llamar variantes sin recalibrar, quedarse con las de muy alta confianza, usarlas como `--known-sites`, recalibrar, y volver a llamar.

Requisitos para que `BaseRecalibrator` funcione (los errores más comunes en la práctica):

- BAM **ordenado por coordenada** y con **read group** (`@RG`).
- Referencia con `.fai` y `.dict` (sección 3.1).
- VCF de sitios conocidos **indexado** (`gatk IndexFeatureFile`) y con **los mismos nombres de contigs** que la referencia (el clásico problema `chr1` vs `1`).

### 6.3 Sitios conocidos para nuestro ejemplo

No existe un "dbSNP" para nuestro mtDNA simulado. Usaremos como *known-sites* el VCF que generó `dwgsim` con las mutaciones que puso en la muestra. **Ojo, es una simplificación de clase**: en un análisis real los sitios conocidos son variantes *de la población* (dbSNP, 1000 Genomas), no las variantes de la propia muestra. Lo usamos porque nos permite ver con claridad el efecto del enmascaramiento. Y no arruina lo que viene: en la próxima clase seguiremos comparando lo que encuentre el llamador contra este mismo archivo.

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase5

cp fastq/mtDNA_sample.mutations.vcf sitios_conocidos.vcf

echo "--- Sitios conocidos (sin las lineas de encabezado ##) ---"
grep -v '^##' sitios_conocidos.vcf | cut -f1-5

echo
echo "--- Indexando el VCF (GATK crea un .idx) ---"
gatk --java-options "-Xmx2g" IndexFeatureFile -I sitios_conocidos.vcf 2>&1 | tail -2
ls -la sitios_conocidos.vcf*

### 6.4 Paso 1: `BaseRecalibrator`

Como entrada usamos el BAM con duplicados marcados de la sección 5 (`mtDNA_sample.markdup.bam`). `BaseRecalibrator` **excluye** automáticamente las lecturas marcadas como duplicado (lo veremos en su log: *NotDuplicateReadFilter*), para que una misma molécula no cuente varias veces en el modelo de error.

```
gatk BaseRecalibrator -I entrada.bam -R referencia.fasta --known-sites sitios.vcf -O tabla.table
```

| Argumento | Para qué sirve |
|---|---|
| `-I` | BAM de entrada |
| `-R` | Referencia (con `.fai` y `.dict`) |
| `--known-sites` | VCF de variantes conocidas a enmascarar. Se puede repetir para varios VCFs |
| `-O` | Tabla de recalibración de salida (texto) |

Fíjense en que `BaseRecalibrator` **no modifica el BAM**: solo produce la tabla. También recuerden que en GATK los argumentos van con doble guion y en minúsculas (`--known-sites`, no `-known-sites`).

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase5/alineamiento

gatk --java-options "-Xmx2g" BaseRecalibrator \
    -I mtDNA_sample.markdup.bam \
    -R ../referencia/chrM.fasta \
    --known-sites ../sitios_conocidos.vcf \
    -O recal_data.table 2> baserecalibrator.log

tail -3 baserecalibrator.log | cut -c1-150
echo
ls -la recal_data.table

### 6.5 Cómo leer la tabla de recalibración

El archivo es un "informe GATK" con varias tablas. Las tres importantes van de lo general a lo específico:

- **`RecalTable0`**: una fila por *read group*. Resumen global: cuántas bases se observaron, cuántos errores, la calidad promedio **reportada** (`EstimatedQReported`) y la calidad **empírica** (`EmpiricalQuality`, calculada a partir de los errores realmente observados).
- **`RecalTable1`**: una fila por calidad reportada. ¿Las bases que el secuenciador dijo "Q30" realmente fallan 1 de cada 1.000?
- **`RecalTable2`**: filas por cada combinación de las covariables ciclo y contexto.

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase5/alineamiento

echo "--- RecalTable0: resumen por read group ---"
sed -n '/^#:GATKTable:RecalTable0:/,/^$/p' recal_data.table | tail -n +2

echo "--- RecalTable1: por calidad reportada (QualityScore) ---"
sed -n '/^#:GATKTable:RecalTable1:/,/^$/p' recal_data.table | tail -n +2

**¿Qué deberían ver?** En `RecalTable0`: unas 399.000 observaciones (2000 parejas × 200 bases, menos las bases de las lecturas duplicadas y las de los sitios conocidos), ≈390 errores, y una calidad empírica de **Q30**. Eso encaja con cómo simulamos los datos: `dwgsim -e 0.001` genera un error de secuenciación de 0,1 % = Q30.

En `RecalTable1`, las calidades **reportadas** van de Q22 a Q39, pero la calidad **empírica** del grueso de las bases (las filas con más observaciones) queda cerca de Q30. Traducción: el simulador asignó a cada base una calidad que varía, pero los errores reales **no dependen de ella**; BQSR lo descubre y "aplana" las calidades hacia lo que realmente ocurre. Los extremos (Q22-Q25 y Q35-Q39) tienen pocas observaciones, y ahí la calidad empírica se queda cerca de la reportada, porque GATK usa un estimador **bayesiano**: con poca evidencia no se aleja de la calidad reportada.

### 6.6 ¿De verdad enmascaró los sitios conocidos?

Podemos comprobarlo con números. `samtools stats` (con `-d`, para que también ignore los duplicados, igual que GATK) cuenta **todos** los *mismatches* del BAM, mientras que la tabla de GATK reporta los **errores** que sí contó. La diferencia debería ser justamente lo que había en los sitios conocidos:

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase5/alineamiento

# a) mismatches totales segun samtools y bases totales alineadas
MISM=$(samtools stats -d mtDNA_sample.markdup.bam | grep "^SN" | grep "mismatches:" | cut -f3)
BASES=$(samtools stats -d mtDNA_sample.markdup.bam | grep "^SN" | grep "bases mapped (cigar):" | cut -f3)

# b) observaciones y errores que conto BaseRecalibrator (RecalTable0)
read OBS ERR <<< $(sed -n '/^#:GATKTable:RecalTable0:/,/^$/p' recal_data.table | awk 'NR==3{print $5, $6}')

# c) cobertura y bases distintas a la referencia justo en los sitios conocidos
grep -v '^#' ../sitios_conocidos.vcf | cut -f1,2 > sitios.txt
read COB ALT <<< $(samtools mpileup -f ../referencia/chrM.fasta -l sitios.txt -Q 0 -q 0 -B mtDNA_sample.markdup.bam 2>/dev/null \
    | awk '{cob+=$4; s=$5; gsub(/\^./,"",s); gsub(/[.,*$]/,"",s); alt+=length(s)} END{print cob, alt}')

echo "Bases alineadas (samtools stats):            $BASES"
echo "Bases que uso BaseRecalibrator (tabla):     $OBS   -> enmascaro $((BASES - OBS))"
echo "Cobertura total en los sitios conocidos:     $COB"
echo
echo "Mismatches totales (samtools stats):         $MISM"
echo "Errores contados por BaseRecalibrator:      ${ERR%.*}   -> descarto $((MISM - ${ERR%.*}))"
echo "Bases no-referencia en los sitios conocidos: $ALT"

Las cifras cuadran: las bases que `BaseRecalibrator` **dejó fuera** son exactamente las que caen en los sitios conocidos, y los mismatches que **descartó** son las variantes reales (no errores) que estaban ahí. Si no hubiéramos entregado `--known-sites`, esas variantes habrían inflado la tasa de error y las calidades habrían quedado subestimadas. *(En datos reales el cuadre no es tan exacto porque BQSR aplica otros filtros: bases de muy baja calidad, extremos de lectura, etc.)*

### 6.7 Paso 2: `ApplyBQSR`

```
gatk ApplyBQSR -I entrada.bam -R referencia.fasta --bqsr-recal-file tabla.table -O salida.bam
```

Aquí sí se genera un BAM nuevo, con las calidades reescritas. Agregamos `--emit-original-quals true`: guarda la calidad **original** de cada base en la etiqueta **`OQ`**, de modo que la recalibración sea reversible y podamos comparar antes/después.

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase5/alineamiento

gatk --java-options "-Xmx2g" ApplyBQSR \
    -I mtDNA_sample.markdup.bam \
    -R ../referencia/chrM.fasta \
    --bqsr-recal-file recal_data.table \
    --emit-original-quals true \
    -O mtDNA_sample.recal.bam 2> applybqsr.log

tail -2 applybqsr.log | cut -c1-150
ls -la mtDNA_sample.recal.bam*

echo
echo "--- Calidades de la primera lectura: antes (OQ) y despues (QUAL) ---"
samtools view mtDNA_sample.recal.bam | head -1 | awk '{
    for (i = 12; i <= NF; i++) if ($i ~ /^OQ:Z:/) oq = substr($i, 6)
    print "antes (OQ):   " oq
    print "despues:      " $11 }'

echo
echo "--- Calidad promedio del BAM (samtools stats) ---"
for b in markdup recal; do
    printf "%-8s " $b; samtools stats mtDNA_sample.$b.bam | grep "average quality" | cut -f2,3
done

Las calidades se ven como letras porque están codificadas en **Phred+33** (ASCII): `?` = Q30, `>` = Q29, `@` = Q31, y así sucesivamente. Antes había una mezcla de valores (`A`, `B`, `C`, `;`, `<`...); después, casi todo cae en `?` y `>` (Q29-Q30), consistente con lo que vimos en la tabla.

### 6.8 Comprobar la recalibración: un segundo `BaseRecalibrator`

La prueba de que quedó bien calibrado: volver a correr `BaseRecalibrator` sobre el BAM **recalibrado**. Ahora la calidad *reportada* (la nueva) debería coincidir con la *empírica*.

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase5/alineamiento

gatk --java-options "-Xmx2g" BaseRecalibrator \
    -I mtDNA_sample.recal.bam \
    -R ../referencia/chrM.fasta \
    --known-sites ../sitios_conocidos.vcf \
    -O recal_data.post.table 2> baserecalibrator2.log

echo "--- DESPUES de la recalibracion: RecalTable1 ---"
sed -n '/^#:GATKTable:RecalTable1:/,/^$/p' recal_data.post.table | tail -n +2

Ahora hay solo dos calidades reportadas (Q29 y Q30) y cada una coincide con su calidad empírica: el modelo quedó **calibrado**. (Para una comparación gráfica antes/después existe `gatk AnalyzeCovariates`, que genera un PDF con gráficos; requiere R con algunos paquetes adicionales, por eso no lo usamos hoy.)

### 6.9 ¿Hay que hacer BQSR siempre?

- La documentación de GATK dice que *"casi siempre"* conviene, y es un paso estándar en los flujos de datos humanos nucleares (WGS/WES).
- Su beneficio depende de la calidad de los datos y de tener buenos sitios conocidos. Los secuenciadores modernos entregan calidades mucho mejor ajustadas que hace unos años, y algunos las agrupan en pocos valores (*binning*, por ejemplo en NovaSeq); por eso se discute cuánto aporta BQSR hoy según el tipo de datos. La documentación de GATK sigue recomendándolo casi siempre, pero es una decisión metodológica que **hay que justificar**, no una receta ciega.
- Sin catálogo de variantes conocidas (organismos no modelo, mtDNA) es difícil hacerlo bien. El flujo de GATK para variantes mitocondriales no incluye BQSR entre sus pasos.
- Requiere bastantes datos para estimar la tabla. Con muy poca cobertura o pocas bases, la tabla es inestable.

> **Para discutir (2 min, en parejas):** (a) Si el cliente les entrega un BAM ya recalibrado con otro flujo, ¿qué pasa si lo recalibran otra vez? ¿Para qué sirve la etiqueta `OQ`? (b) ¿Qué pasaría con las calidades si un tercio de las variantes reales de la muestra **no** estuvieran en `--known-sites`?

---
## 7. ¿Qué es un workflow? (10 min)

Hasta ahora corrimos los pasos uno por uno, a mano, copiando nombres de archivo de una celda a otra. Funciona para aprender, pero en un análisis real es frágil: ¿qué pasa con 50 muestras? ¿Y si dentro de seis meses hay que repetirlo, o lo debe correr otra persona?

### 7.1 Definición

Un **workflow** (flujo de trabajo, también llamado *pipeline*) es una **secuencia definida de pasos de análisis** donde la **salida de un paso es la entrada del siguiente**, descrita de manera explícita y ejecutable, de modo que se pueda repetir con distintos datos obteniendo resultados reproducibles. Es la diferencia entre "ejecuté estos comandos y me funcionó" y "aquí está la receta completa; cualquiera puede repetirla".

Cada workflow tiene:

- **Pasos (tareas):** cada uno ejecuta una herramienta (`bwa mem`, `FixMateInformation`, `BaseRecalibrator`...).
- **Entradas y salidas** de cada paso (archivos), y por tanto **dependencias** entre pasos.
- **Parámetros:** lo que cambia entre corridas (muestra, referencia, número de hebras).
- **Software y versiones** de cada herramienta (nuestro ambiente conda).
- **Recursos:** memoria, hebras, tiempo.

Las dependencias forman un **grafo dirigido sin ciclos (DAG)**. El de nuestro flujo, con dos muestras, es:

```
              ┌─► alinear+ordenar ─► FixMate ─► MarkDup ─► BaseRecalibrator ─► ApplyBQSR ─► BAM final ─┐
  muestra A ──┤                                                                                         │
              └─ (usa: referencia + índices + .dict + known-sites)                                      ├─► (llamado de variantes)
              ┌─► alinear+ordenar ─► FixMate ─► MarkDup ─► BaseRecalibrator ─► ApplyBQSR ─► BAM final ─┘
  muestra B ──┘
```

Las dos ramas son **independientes**: se pueden ejecutar al mismo tiempo. Dentro de cada rama, cada paso **espera** al anterior. Detectar y respetar esto es el trabajo de un buen workflow.

### 7.2 Script vs. gestor de workflows

| | Script de shell (lo que haremos hoy) | Gestor de workflows (Snakemake, Nextflow, WDL/Cromwell, CWL) |
|---|---|---|
| Se describe... | **cómo** ejecutar: la lista de comandos en orden | **qué** produce cada paso y de qué depende; el gestor deduce el orden |
| Reanudar tras un fallo | Hay que programarlo a mano (lo haremos con una comprobación simple) | Automático: solo repite lo que falta |
| Paralelizar muestras | A mano (`&`, `xargs`, un `for` en el clúster) | Automático, y se adapta a clústeres y a la nube |
| Software | El que esté en el PATH | Un ambiente conda o contenedor **por paso** |
| Curva de aprendizaje | Baja | Media-alta |
| Cuándo conviene | Pocas muestras, análisis puntual, prototipos | Muchas muestras, análisis que se repetirán, trabajo en equipo |

Los gestores más usados en bioinformática son **Snakemake** (basado en Python), **Nextflow** (con una gran colección de flujos ya hechos y revisados por la comunidad, *nf-core*) y **WDL/Cromwell** (el lenguaje con que el Broad Institute publica los flujos oficiales de GATK). No los veremos en profundidad en este curso, pero conviene saber que **el script bien escrito de hoy es el paso previo natural**: cada regla de un gestor es, en el fondo, uno de los comandos que ya escribimos.

Solo para ver la idea (**no lo ejecutaremos**), así se describirían nuestros primeros pasos en Snakemake. Fíjense en que no se escribe el orden: se declara que cada archivo `output` se construye a partir de sus `input`, y Snakemake infiere la cadena:

```python
# Snakefile (ilustrativo)
MUESTRAS = ["muestraA", "muestraB"]

rule all:                                     # lo que queremos obtener al final
    input: expand("resultados/{m}.fixmate.bam", m=MUESTRAS)

rule alinear:
    input:  ref="referencia/chrM.fasta", r1="fastq/{m}_R1.fastq.gz", r2="fastq/{m}_R2.fastq.gz"
    output: "resultados/{m}.sorted.bam"
    threads: 2
    shell:  "bwa mem -t {threads} -R '@RG\\tID:{wildcards.m}\\tSM:{wildcards.m}\\tPL:ILLUMINA' "
            "{input.ref} {input.r1} {input.r2} | samtools sort -@ {threads} -o {output} -"

rule fixmate:
    input:  "resultados/{m}.sorted.bam"
    output: "resultados/{m}.fixmate.bam"
    shell:  "picard FixMateInformation -I {input} -O {output} --ADD_MATE_CIGAR true"
```

Si `muestraA.sorted.bam` ya existe y sus entradas no cambiaron, Snakemake no lo vuelve a generar. Ese es el comportamiento que en nuestro script bash vamos a imitar a mano con un `if`.

---
## 8. Buenas prácticas para escribir un script de alineamiento y mejoramiento (10 min)

Un script que "funciona en mi computador" no es suficiente. Un buen script de análisis es **reproducible** (da el mismo resultado si se repite), **legible** (otra persona lo entiende), y **robusto** (falla rápido y con un mensaje claro cuando algo está mal, en vez de continuar con datos corruptos). Estas son las prácticas que aplicaremos en el script de la sección 9, agrupadas por objetivo:

**Robustez: que falle rápido y claro**

1. **Modo estricto:** `set -euo pipefail` al inicio. `-e` aborta ante cualquier error, `-u` ante variables no definidas (evita el temido `rm -rf $DIR/*` con `$DIR` vacío), `-o pipefail` hace que un pipe falle si **cualquier** etapa falla (sin él, `bwa mem | samtools sort` reportaría éxito aunque `bwa` haya muerto).
2. **Validar entradas antes de empezar:** que existan los archivos, que las herramientas estén en el PATH, que los argumentos sean los esperados. Es mejor fallar en el segundo 1 que a las 3 horas.
3. **Mensajes de error útiles:** decir *qué* falló y *dónde*, no solo terminar (`trap ... ERR`).
4. **Escritura atómica:** cada paso escribe a un archivo temporal y **solo al terminar bien** lo renombra. Así una interrupción nunca deja un BAM a medias que el siguiente intento tome por bueno.

**Reproducibilidad**

5. **Nada "quemado" en el código:** rutas, nombre de muestra, hebras y memoria son **parámetros** (argumentos), no valores escritos dentro del script. El mismo script sirve para todas las muestras.
6. **Registrar todo:** un archivo de **log** con fecha y hora de cada paso, y las **versiones** de cada herramienta. Dentro de un año serán lo único que les diga cómo se generó un BAM.
7. **Ambiente controlado:** ejecutar dentro de un ambiente conda (idealmente exportado con `conda env export`). La **misma referencia** y el mismo `.dict` en todos los pasos.
8. **Datos de entrada intactos:** los FASTQ originales nunca se modifican ni se borran; los resultados van a un directorio aparte.
9. **Control de versiones:** guardar el script en `git`. (Anotar en el log qué versión del script se usó.)

**Eficiencia y buen uso del computador**

10. **Pipes en vez de intermedios grandes:** `bwa mem | samtools sort` evita escribir un SAM de decenas de GB. Pero cuidado: solo se encadena lo que no necesitaremos por separado.
11. **Idempotencia y reanudación:** si el resultado de un paso ya existe, se salta. Permite reanudar tras un fallo sin rehacer horas de cómputo.
12. **Limpieza:** borrar los BAM intermedios cuando ya no se necesitan (con una opción para conservarlos al depurar) y usar un directorio temporal propio (`--TMP_DIR`, `--tmp-dir`) en vez de llenar `/tmp`.
13. **Recursos explícitos:** número de hebras (`-t`) y memoria de Java (`-Xmx`) como parámetros. Un `-Xmx` mayor que la RAM disponible mata el proceso.
14. **En el clúster:** los trabajos largos no se corren en el nodo de entrada (*login node*) ni en una terminal que se pueda cerrar; se lanzan con `tmux`/`screen`/`nohup` o con el gestor de colas del clúster.

**Legibilidad**

15. **Comentar el *por qué***, no el *qué*. Encabezado que explique qué hace el script y cómo se usa; opción `-h` con ayuda.
16. **Comillas en las variables:** `"$ARCHIVO"`, no `$ARCHIVO` (si el nombre tiene espacios, todo se rompe).
17. **Nombres consistentes** y con estructura predecible (`{muestra}.sorted.bam`, `{muestra}.final.bam`...).

**Validar el resultado (no solo que el script "termine")**

18. Comprobar la salida de cada etapa: `samtools quickcheck`, `flagstat`, `ValidateSamFile`. Un script que termina sin error puede haber producido un BAM inútil.
19. **Probar primero con datos pequeños** (como nuestro mtDNA) antes de lanzarlo sobre 50 muestras de genoma completo.

Un orden de pasos clásico y su justificación, para el flujo de hoy: **alinear → ordenar → FixMate → (MarkDuplicates) → BaseRecalibrator → ApplyBQSR → validar.** `FixMate` va justo después de alinear y ordenar porque, desde ahí en adelante, todas las herramientas deben recibir un BAM coherente (y algunas, como `MarkDuplicatesWithMateCigar`, exigen la etiqueta `MC`); `MarkDuplicates` va antes de BQSR porque el modelo debe aprender el error solo de las lecturas "definitivas": `BaseRecalibrator` **excluye** las lecturas marcadas como duplicado.

> **Sobre MarkDuplicates:** en WGS/WES estándar conviene marcarlos. En **amplicones o PCR de largo alcance** (por ejemplo, mtDNA amplificado por long-range PCR) **no** se debe hacer: casi todas las lecturas "duplican" posiciones por diseño y se descartaría cobertura legítima. Por eso el script tendrá una opción para omitirlo (`-D`).

---
## 9. El script: `alinear_y_mejorar.sh` (20 min)

Vamos a juntar todo en **un solo script** que, para una muestra pareada, hace: `bwa mem` + ordenar → `FixMateInformation` → `MarkDuplicates` (opcional) → `BaseRecalibrator` + `ApplyBQSR` → validación. Léanlo con calma: **cada práctica de la sección 8 aparece aquí**, y los comentarios indican cuál. Escribimos el archivo en `~/bioinfo/clase5/scripts/`.

In [ ]:
%%bash
cat > ~/bioinfo/clase5/scripts/alinear_y_mejorar.sh << 'SCRIPT_EOF'
#!/usr/bin/env bash
#
# alinear_y_mejorar.sh
# FASTQ pareado  ->  BAM "listo para analisis"
#
# Pasos:  bwa mem + sort  ->  FixMateInformation  ->  MarkDuplicates (opcional)
#         ->  BaseRecalibrator + ApplyBQSR  ->  validacion
#
# Requiere en el PATH: bwa, samtools, picard, gatk
#
set -euo pipefail          # abortar ante errores, variables sin definir y fallos dentro de pipes
VERSION="1.0"

# ---------------------------------------------------------------- 1. Ayuda
usage() {
    cat <<USO
alinear_y_mejorar.sh v${VERSION}

Uso: $(basename "$0") -s MUESTRA -r REF.fasta -1 R1.fastq.gz -2 R2.fastq.gz -k SITIOS.vcf [opciones]

Obligatorios:
  -s  nombre de la muestra (se usa en el read group y en los archivos de salida)
  -r  referencia FASTA (con o sin sus indices)
  -1  FASTQ read 1
  -2  FASTQ read 2
  -k  VCF de sitios variables conocidos (para BQSR)

Opcionales:
  -o  directorio de salida            (por defecto: resultados)
  -t  numero de hebras                (por defecto: 2)
  -m  memoria de Java en GB           (por defecto: 4)
  -D  NO marcar duplicados            (usar en amplicones / PCR largo)
  -K  conservar los BAM intermedios
  -h  mostrar esta ayuda
USO
}

# ---------------------------------------------------------------- 2. Parametros
SAMPLE=""; REF=""; R1=""; R2=""; KNOWN=""
OUT="resultados"; THREADS=2; MEM=4; MARKDUP=1; KEEP=0

while getopts "s:r:1:2:k:o:t:m:DKh" opt; do
    case "$opt" in
        s) SAMPLE="$OPTARG" ;;
        r) REF="$OPTARG" ;;
        1) R1="$OPTARG" ;;
        2) R2="$OPTARG" ;;
        k) KNOWN="$OPTARG" ;;
        o) OUT="$OPTARG" ;;
        t) THREADS="$OPTARG" ;;
        m) MEM="$OPTARG" ;;
        D) MARKDUP=0 ;;
        K) KEEP=1 ;;
        h) usage; exit 0 ;;
        *) usage; exit 1 ;;
    esac
done

if [[ -z "$SAMPLE" || -z "$REF" || -z "$R1" || -z "$R2" || -z "$KNOWN" ]]; then
    echo "ERROR: faltan argumentos obligatorios." >&2
    usage; exit 1
fi

# ---------------------------------------------------------------- 3. Log
mkdir -p "$OUT" "$OUT/tmp"
LOG="$OUT/${SAMPLE}.log"
exec 3>&1                                    # fd 3 = la pantalla
exec >>"$LOG" 2>&1                           # el detalle de bwa/picard/gatk va SOLO al log (pantalla limpia)
log() { local m; m="[$(date '+%F %T')] $*"; echo "$m"; echo "$m" >&3; }        # mensaje a log y pantalla
err() { local m; m="[$(date '+%F %T')] ERROR: $*"; echo "$m"; echo "$m" >&3; } # idem, para errores
trap 'err "el script fallo cerca de la linea $LINENO. Ultimas lineas del log:"; tail -n 8 "$LOG" >&3' ERR

# ---------------------------------------------------------------- 4. Chequeos previos
for cmd in bwa samtools picard gatk; do
    command -v "$cmd" >/dev/null 2>&1 || { err "'$cmd' no esta en el PATH. Active el ambiente conda."; exit 1; }
done
for f in "$REF" "$R1" "$R2" "$KNOWN"; do
    [[ -s "$f" ]] || { err "no existe o esta vacio: $f"; exit 1; }
done

log "=== alinear_y_mejorar.sh v${VERSION} | muestra: ${SAMPLE} ==="
log "Versiones -> bwa $(bwa 2>&1 | grep -m1 Version | cut -d' ' -f2) | samtools $(samtools --version | head -1 | cut -d' ' -f2)" \
    "| picard $(picard FixMateInformation --version 2>&1 | grep -m1 -o '[0-9][0-9.]*')" \
    "| gatk $(gatk --version 2>&1 | grep -m1 -o 'v[0-9][0-9.]*')"

# Archivos de salida
BAM_ORD="$OUT/${SAMPLE}.sorted.bam"
BAM_FIX="$OUT/${SAMPLE}.fixmate.bam"
BAM_DUP="$OUT/${SAMPLE}.markdup.bam"
TABLA="$OUT/${SAMPLE}.recal_data.table"
BAM_FINAL="$OUT/${SAMPLE}.final.bam"

# Si el resultado final ya existe, no hay nada que hacer (el script es idempotente)
if [[ -s "$BAM_FINAL" ]]; then
    log "$BAM_FINAL ya existe: nada que hacer. Borrelo para volver a correr."
    exit 0
fi

# Escritura "atomica": cada paso escribe a un .tmp.bam y solo al terminar bien se renombra.
# Asi un paso interrumpido nunca deja un BAM a medias que el siguiente intento tome por bueno.
tmp() { echo "${1%.bam}.tmp.bam"; }

# ---------------------------------------------------------------- 5. Indices de la referencia
[[ -s "${REF}.bwt" ]]           || { log "Indexando referencia (bwa)";      bwa index "$REF"; }
[[ -s "${REF}.fai" ]]           || { log "Indexando referencia (samtools)"; samtools faidx "$REF"; }
[[ -s "${REF%.*}.dict" ]]       || { log "Creando diccionario (GATK)";      gatk --java-options "-Xmx${MEM}g" CreateSequenceDictionary -R "$REF"; }
[[ -s "${KNOWN}.idx" || -s "${KNOWN}.tbi" ]] || { log "Indexando sitios conocidos"; gatk --java-options "-Xmx${MEM}g" IndexFeatureFile -I "$KNOWN"; }

# ---------------------------------------------------------------- 6. Pasos del flujo
# 6.1 Alineamiento + orden por coordenada (en un solo pipe: no se escribe un SAM intermedio)
if [[ -s "$BAM_ORD" ]]; then
    log "[1/5] SALTADO: alineamiento ya existe"
else
    log "[1/5] Alineando con bwa mem y ordenando"
    bwa mem -t "$THREADS" -R "@RG\tID:${SAMPLE}\tSM:${SAMPLE}\tLB:${SAMPLE}\tPL:ILLUMINA" \
        "$REF" "$R1" "$R2" \
      | samtools sort -@ "$THREADS" -O bam -o "$(tmp "$BAM_ORD")" -
    mv "$(tmp "$BAM_ORD")" "$BAM_ORD"
fi

# 6.2 Sincronizar la informacion de las parejas (mates) y agregar la etiqueta MC
if [[ -s "$BAM_FIX" ]]; then
    log "[2/5] SALTADO: FixMateInformation ya existe"
else
    log "[2/5] Picard FixMateInformation"
    picard -Xmx${MEM}g FixMateInformation \
        -I "$BAM_ORD" -O "$(tmp "$BAM_FIX")" \
        --ADD_MATE_CIGAR true --TMP_DIR "$OUT/tmp"
    mv "$(tmp "$BAM_FIX")" "$BAM_FIX"
fi
ULTIMO="$BAM_FIX"

# 6.3 Marcar duplicados (opcional)
if [[ "$MARKDUP" -eq 1 ]]; then
    if [[ -s "$BAM_DUP" ]]; then
        log "[3/5] SALTADO: MarkDuplicates ya existe"
    else
        log "[3/5] Picard MarkDuplicates"
        picard -Xmx${MEM}g MarkDuplicates \
            -I "$BAM_FIX" -O "$(tmp "$BAM_DUP")" \
            -M "$OUT/${SAMPLE}.dup_metrics.txt" --TMP_DIR "$OUT/tmp"
        mv "$(tmp "$BAM_DUP")" "$BAM_DUP"
    fi
    ULTIMO="$BAM_DUP"
else
    log "[3/5] OMITIDO: MarkDuplicates (opcion -D)"
fi

# 6.4 BQSR paso 1: construir el modelo de error
if [[ -s "$TABLA" ]]; then
    log "[4/5] SALTADO: tabla de recalibracion ya existe"
else
    log "[4/5] GATK BaseRecalibrator"
    gatk --java-options "-Xmx${MEM}g" BaseRecalibrator \
        -I "$ULTIMO" -R "$REF" --known-sites "$KNOWN" \
        -O "${TABLA}.tmp" --tmp-dir "$OUT/tmp"
    mv "${TABLA}.tmp" "$TABLA"
fi

# 6.5 BQSR paso 2: aplicar el modelo (conservando las calidades originales en la etiqueta OQ)
log "[5/5] GATK ApplyBQSR"
gatk --java-options "-Xmx${MEM}g" ApplyBQSR \
    -I "$ULTIMO" -R "$REF" --bqsr-recal-file "$TABLA" \
    --emit-original-quals true \
    -O "$(tmp "$BAM_FINAL")" --tmp-dir "$OUT/tmp"
mv "$(tmp "$BAM_FINAL")" "$BAM_FINAL"
[[ -s "${BAM_FINAL%.bam}.tmp.bai" ]] && mv "${BAM_FINAL%.bam}.tmp.bai" "${BAM_FINAL}.bai"
[[ -s "${BAM_FINAL}.bai" ]] || samtools index "$BAM_FINAL"

# ---------------------------------------------------------------- 7. Validacion del resultado
log "Validando el BAM final"
samtools quickcheck -v "$BAM_FINAL"
VALIDACION=$(picard -Xmx${MEM}g ValidateSamFile -I "$BAM_FINAL" -R "$REF" --MODE SUMMARY 2>&1 || true)
echo "$VALIDACION"                                   # (queda en el log)
if echo "$VALIDACION" | grep -q "No errors found"; then
    log "ValidateSamFile: sin errores"
else
    err "ValidateSamFile encontro problemas (detalle en $LOG)"; exit 1
fi
samtools flagstat "$BAM_FINAL" > "$OUT/${SAMPLE}.final.flagstat.txt"

# ---------------------------------------------------------------- 8. Limpieza
rm -rf "$OUT/tmp"
if [[ "$KEEP" -eq 0 ]]; then
    log "Borrando BAM intermedios (use -K para conservarlos)"
    rm -f "$BAM_ORD" "${BAM_ORD}.bai" "$BAM_FIX" "${BAM_FIX%.bam}.bai" "$BAM_DUP" "${BAM_DUP%.bam}.bai"
fi

log "LISTO -> $BAM_FINAL"
SCRIPT_EOF

chmod +x ~/bioinfo/clase5/scripts/alinear_y_mejorar.sh
echo "Script guardado:"
ls -la ~/bioinfo/clase5/scripts/

echo
echo "--- Comprobacion de sintaxis (bash -n) ---"
bash -n ~/bioinfo/clase5/scripts/alinear_y_mejorar.sh && echo "sintaxis OK"

### 9.1 Recorrido por el script

| Sección | Qué hace | Práctica (§8) |
|---|---|---|
| Encabezado | `set -euo pipefail`: aborta ante errores, variables sin definir y fallos dentro de pipes | 1 |
| 1. Ayuda | Función `usage` con `-h` | 15 |
| 2. Parámetros | `getopts` lee `-s -r -1 -2 -k -o -t -m -D -K`; ninguna ruta está "quemada" | 5 |
| 3. Log | `exec >>"$LOG" 2>&1` manda el detalle de `bwa`/`picard`/`gatk` **solo al log** (pantalla limpia); las funciones `log` y `err` escriben a pantalla **y** log; `trap ... ERR` avisa en qué línea falló y muestra las últimas líneas del log | 3, 6 |
| 4. Chequeos | ¿están las 4 herramientas? ¿existen los archivos de entrada? Registra las versiones | 2, 6 |
| Idempotencia | Si el BAM final ya existe, termina sin hacer nada | 11 |
| `tmp()` | Cada paso escribe a `*.tmp.bam` y solo al terminar bien se renombra con `mv` | 4 |
| 5. Índices | Crea `.bwt`, `.fai`, `.dict` y el índice del VCF **solo si faltan** | 11 |
| 6. Pasos | Cada paso comprueba "¿ya existe mi salida?" y, si sí, se salta | 10, 11 |
| 7. Validación | `quickcheck`, `ValidateSamFile` (el script **lee su resultado** y aborta si no dice "No errors found") y `flagstat` sobre el BAM **final** | 18 |
| 8. Limpieza | Borra intermedios (salvo con `-K`) y el directorio temporal | 12 |

Ahora lo ejecutamos sobre nuestra muestra. Como el detalle de cada herramienta queda en el log, en pantalla solo verán los pasos. Con estos datos tarda unos 20-30 segundos:

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase5

./scripts/alinear_y_mejorar.sh \
    -s mtDNA_sample \
    -r referencia/chrM.fasta \
    -1 fastq/mtDNA_sample_R1.fastq.gz -2 fastq/mtDNA_sample_R2.fastq.gz \
    -k sitios_conocidos.vcf \
    -o resultados -t 2 -m 4

echo
echo "--- Contenido de resultados/ ---"
ls -la resultados

Revisemos qué dejó el script. El **log** es la "bitácora" de la corrida (con todo el detalle de cada herramienta; aquí solo mostramos las líneas con fecha); el `flagstat` y la tabla de recalibración son los resultados que se conservan; y el BAM final trae las calidades recalibradas (con `OQ`) y las parejas sincronizadas (con `MC`).

Noten que el script repite lo que hicimos a mano en las secciones 4 a 6, en el mismo orden: el detalle de `MarkDuplicates` (≈2 parejas duplicadas por casualidad, como en la sección 5) quedó en `dup_metrics.txt`, y la tabla de recalibración es la misma que calculamos a mano.

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase5/resultados

echo "--- Log (solo las lineas con fecha) ---"
grep '^\[20' mtDNA_sample.log

echo
echo "--- flagstat del BAM final ---"
head -5 mtDNA_sample.final.flagstat.txt

echo
echo "--- Metricas de duplicados (Picard MarkDuplicates) ---"
grep -A1 '^LIBRARY' mtDNA_sample.dup_metrics.txt | cut -f1-3,7-9

echo
echo "--- Una lectura del BAM final: etiquetas OQ, MC y MQ presentes ---"
samtools view mtDNA_sample.final.bam | head -1 | cut -f12- | tr '\t' '\n' | cut -c1-30

### 9.2 Probar la robustez: idempotencia y errores

Dos pruebas que **todo script debería pasar**: (1) si lo corro dos veces, la segunda no rehace nada; (2) si le doy algo mal, debe fallar rápido, con un mensaje claro y con **código de salida distinto de cero** (los gestores y los clústeres usan ese código para saber si el trabajo salió bien).

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase5

echo "=== Prueba 1: correrlo de nuevo con los mismos parametros ==="
./scripts/alinear_y_mejorar.sh -s mtDNA_sample -r referencia/chrM.fasta \
    -1 fastq/mtDNA_sample_R1.fastq.gz -2 fastq/mtDNA_sample_R2.fastq.gz \
    -k sitios_conocidos.vcf -o resultados 2>&1 | tail -1

echo
echo "=== Prueba 2: archivo de entrada que no existe ==="
./scripts/alinear_y_mejorar.sh -s prueba -r referencia/chrM.fasta \
    -1 fastq/NO_EXISTE_R1.fastq.gz -2 fastq/mtDNA_sample_R2.fastq.gz \
    -k sitios_conocidos.vcf -o /tmp/prueba_clase5 > /tmp/salida_prueba2.txt 2>&1
echo "codigo de salida: $?"
tail -1 /tmp/salida_prueba2.txt

echo
echo "=== Prueba 3: falta un argumento obligatorio (-k) ==="
./scripts/alinear_y_mejorar.sh -s prueba -r referencia/chrM.fasta \
    -1 fastq/mtDNA_sample_R1.fastq.gz -2 fastq/mtDNA_sample_R2.fastq.gz \
    -o /tmp/prueba_clase5 > /tmp/salida_prueba3.txt 2>&1
echo "codigo de salida: $?"
head -2 /tmp/salida_prueba3.txt

---
## 10. Ejercicio integrador (en parejas, 15 min)

Van a usar el script con **su propia muestra**, y luego a **mejorarlo**.

1. **Simulen** un nuevo set de lecturas con una semilla (`-z`) distinta a la del ejemplo (por ejemplo, los últimos dígitos del RUT de alguno de los dos), con otro nombre de muestra (`mtDNA_pareja`). Guarden los FASTQ en `~/bioinfo/clase5/fastq/` (parámetros de `dwgsim`: los mismos de la sección 3, cambiando `-z` y el nombre de salida; recuerden renombrar a `_R1`/`_R2` y borrar el archivo `bfast`).
2. **Corran el script** sobre su muestra. Como known-sites usen el VCF de mutaciones que generó `dwgsim` para **su** muestra (`mtDNA_pareja.mutations.vcf`), no el de `mtDNA_sample`. (¿Qué pasaría si usaran el equivocado?)
3. **Interpreten** su resultado: abran `resultados/mtDNA_pareja.recal_data.table` y respondan (a) ¿cuál es la calidad reportada promedio y la empírica (`RecalTable0`)? ¿La tasa de error que simularon (`-e 0.001`) coincide con la calidad empírica? (b) ¿Cuántas lecturas hay en su BAM final versus en el FASTQ (`samtools flagstat`)? ¿Cuántas quedaron marcadas como duplicado y por qué ese número no es cero? ¿Se eliminaron del BAM?
4. **Mejoren el script:** hoy valida que los archivos existan, pero **no valida `-t` ni `-m`**. Agréguenle una validación que aborte con un mensaje claro si alguno **no es un número entero positivo**. *Pista:* en bash, `[[ "$THREADS" =~ ^[1-9][0-9]*$ ]]` comprueba eso. Pruébenla con `-t abc` y `-m 0`. Copien el script mejorado como `scripts/alinear_y_mejorar_v2.sh` (¡no sobrescriban el original!).
5. **Desafío (si les alcanza el tiempo):** corran el script para **ambas muestras** (`mtDNA_sample` y `mtDNA_pareja`) con un `for`, sin copiar y pegar el comando dos veces. ¿Qué parte del comando cambia entre muestras? ¿Qué parte de la sección 8 hace posible esto?

No hay una única forma correcta de resolverlo. Lo que importa es que puedan explicar **qué hace cada línea que agregaron y por qué**.

---
## 11. Cierre y resumen

### Conceptos y comandos vistos hoy

```
Instalacion:     conda create -n mejoramiento bwa samtools picard gatk4
                 (manual: picard.jar + gatk-X.zip, requieren Java 17+)
Referencia:      samtools faidx, bwa index, picard CreateSequenceDictionary (.dict)
Mates:           picard FixMateInformation -I in.bam -O out.bam --ADD_MATE_CIGAR true
Validar BAM:     picard ValidateSamFile -I x.bam -R ref.fasta --MODE SUMMARY
Duplicados:      picard MarkDuplicates -I in.bam -O out.bam -M metricas.txt
                 (samtools view -f 1024 / -F 1024 / -c para contarlos; no usar en amplicones)
BQSR:            gatk IndexFeatureFile -I sitios.vcf
                 gatk BaseRecalibrator -I in.bam -R ref.fasta --known-sites sitios.vcf -O tabla.table
                 gatk ApplyBQSR -I in.bam -R ref.fasta --bqsr-recal-file tabla.table
                              --emit-original-quals true -O out.bam
Memoria Java:    picard -Xmx4g ...   |   gatk --java-options "-Xmx4g" ...
Script:          set -euo pipefail, getopts, log + trap, chequeos previos,
                 escritura atomica (.tmp), idempotencia, limpieza, validacion final
```

### Ideas clave

- `FixMateInformation` **sincroniza** los datos de las parejas y agrega `MC`; se usa cuando alguna herramienta modificó un registro sin actualizar a su pareja, y conviene ejecutarlo temprano en el flujo.
- `MarkDuplicates` **marca** (flag `0x400`, sin eliminar) las parejas que provienen de la misma molécula, mirando dónde comienzan y terminan sus dos lecturas dentro de una misma librería (`LB`). Guarda un archivo de **métricas**, y **no debe usarse en amplicones**.
- BQSR **modela el error real** de las calidades base (por read group, calidad reportada, ciclo y contexto) usando sitios variables conocidos para no confundir variantes reales con errores.
- Un **workflow** es una receta explícita y repetible: pasos, entradas/salidas, dependencias y ambiente. Un script bien escrito es su versión más simple.

### Lo que dejamos listo para la próxima clase

- El ambiente `mejoramiento` (bwa, samtools, picard, gatk4, dwgsim) creado y verificado.
- Un BAM **listo para análisis** de cada muestra: `~/bioinfo/clase5/resultados/mtDNA_sample.final.bam` (y el de su pareja) con su índice `.bai`.
- El script `~/bioinfo/clase5/scripts/alinear_y_mejorar.sh`, que pueden reutilizar en sus propios proyectos.
- El archivo de mutaciones de `dwgsim`, que **sigue sin haber sido comparado** contra lo que encuentre un llamador de variantes real.

### Próxima clase: llamado de variantes

Partiremos exactamente de estos BAM finales para pasar de la evidencia en el BAM a un **VCF** formal, con genotipos y medidas de confianza, y compararemos el resultado contra el archivo de mutaciones simuladas.

**Tarea para antes de la próxima clase:**

1. Verifiquen que existen `resultados/mtDNA_sample.final.bam` y `resultados/mtDNA_pareja.final.bam` (con sus `.bai`).
2. Terminen el ejercicio integrador si no alcanzaron (en particular la validación de `-t` y `-m`).
3. Lean con calma la sección 8 y elijan **tres** prácticas que les parezcan más importantes para su propio trabajo; anoten en una línea por qué.
4. Lean el artículo de GATK sobre BQSR (referencias abajo) y anoten una pregunta o duda sobre algo que no les haya quedado claro.

### Referencias

- GATK — [Base Quality Score Recalibration (BQSR)](https://gatk.broadinstitute.org/hc/en-us/articles/360035890531-Base-Quality-Score-Recalibration-BQSR)
- GATK — [BaseRecalibrator](https://gatk.broadinstitute.org/hc/en-us/articles/9570376886683-BaseRecalibrator)
- GATK — [FixMateInformation (Picard)](https://gatk.broadinstitute.org/hc/en-us/articles/360037061172-FixMateInformation-Picard)
- Bioconda — [gatk4](https://bioconda.github.io/recipes/gatk4/README.html) y [picard](https://bioconda.github.io/recipes/picard/README.html)
- Descargas — [GATK releases](https://github.com/broadinstitute/gatk/releases) y [Picard releases](https://github.com/broadinstitute/picard/releases)
- Gestores de workflows — [Snakemake](https://snakemake.github.io), [Nextflow](https://www.nextflow.io) y [nf-core](https://nf-co.re)